In [52]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

seed= 42

In [53]:
df= pd.read_csv('data.csv')

In [54]:
# Subtask 1

answer1= []

answer1.append({
    'subtaskID': 1,
    'datapointID': 1,
    'answer': len(df[
        (df['location'] == 'The Old Library') & 
        (df['tech_assist'] == 1)
        ])
})

answer1= pd.DataFrame(answer1)
answer1.head()

,subtaskID,datapointID,answer
0,1,1,304


In [55]:
# Subtask 2

answer2= []

df_subtask2= df[
    (df['location'] == 'Crystal Cave') & 
    (df['tech_assist'] == 1) &
    ((df['puzzle_type']== 'Logic') | (df['puzzle_type']== 'Sequence'))
    ]

answer2.append({
    'subtaskID': 2,
    'datapointID': 1,
    'answer': round(df_subtask2['skill_level'].mean()+  df_subtask2['teamwork_score'].mean(), 2)
})

answer2= pd.DataFrame(answer2)
answer2.head()

,subtaskID,datapointID,answer
0,2,1,115.24


In [56]:
# Subtask 3

X=df.drop(columns=['RoundID', 'style'])

train_df= df[df['style'] != -1].copy()
test_df= df[df['style'] == -1].copy()

X_train= train_df.drop(columns=['RoundID', 'style'])
y_train= train_df['style']

X_test= train_df.drop(columns=['RoundID', 'style'])

In [57]:
from catboost import CatBoostClassifier

cat_cols= X_train.select_dtypes(include= ['object', 'string']).columns.to_list()

model= CatBoostClassifier(
    iterations=3000,
    learning_rate=0.03,
    depth=6,
    auto_class_weights='Balanced',
    eval_metric='Accuracy',
    cat_features= cat_cols,
    early_stopping_rounds=200,
    random_state= seed
)

model.fit(X_train, y_train, verbose=200)

predictions = model.predict(X)

answer3 = pd.DataFrame([{
    'subtaskID': 3,
    'datapointID': id_,
    'answer': f'{pred[0]:.0f}'
} for id_, pred in zip(df['RoundID'], predictions)])

answer3.head(5)


0:	learn: 0.9500000	total: 16.2ms	remaining: 48.4s
200:	learn: 1.0000000	total: 3.2s	remaining: 44.5s
400:	learn: 1.0000000	total: 6.57s	remaining: 42.6s
600:	learn: 1.0000000	total: 9.99s	remaining: 39.9s
800:	learn: 1.0000000	total: 13.4s	remaining: 36.7s
1000:	learn: 1.0000000	total: 16.8s	remaining: 33.6s
1200:	learn: 1.0000000	total: 20.3s	remaining: 30.4s
1400:	learn: 1.0000000	total: 23.8s	remaining: 27.2s
1600:	learn: 1.0000000	total: 27.3s	remaining: 23.9s
1800:	learn: 1.0000000	total: 30.7s	remaining: 20.5s
2000:	learn: 1.0000000	total: 34.1s	remaining: 17s
2200:	learn: 1.0000000	total: 37.5s	remaining: 13.6s
2400:	learn: 1.0000000	total: 40.9s	remaining: 10.2s
2600:	learn: 1.0000000	total: 44.3s	remaining: 6.79s
2800:	learn: 1.0000000	total: 47.6s	remaining: 3.38s
2999:	learn: 1.0000000	total: 51s	remaining: 0us


,subtaskID,datapointID,answer
0,3,0,0
1,3,1,0
2,3,2,0
3,3,3,0
4,3,4,0


In [58]:
answer= pd.concat([answer1, answer2, answer3])
answer.to_csv('submission.csv', index= False)